# Titanic — Kaggle 実行用ランナー

GitHub のリポジトリを clone して `src/` を読み込むだけの薄いノートブック。
ロジックはすべて GitHub 側にあるので、ここを編集する必要はほぼない。

**事前準備**
1. 右メニュー **Add Input** → Competitions から `Titanic` を追加
2. **Settings → Internet** を On にする（clone に必要）

リポジトリは public なので認証設定は不要。

## 1. GitHub からコードを取得

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = "ekitaiNORI1122/kaggle-titanic"
# public リポジトリなので認証は不要。
# /kaggle/working ではなく /tmp に clone しているのは、/kaggle/working が
# ノートブックの出力として丸ごと保存され、無駄に容量を食うため。
CLONE_DIR = Path("/tmp/repo")

if CLONE_DIR.exists():
    subprocess.run(["rm", "-rf", str(CLONE_DIR)], check=True)
subprocess.run(
    ["git", "clone", "--depth", "1",
     f"https://github.com/{REPO}.git", str(CLONE_DIR)],
    check=True,
)

if str(CLONE_DIR) not in sys.path:
    sys.path.insert(0, str(CLONE_DIR))
print("clone 完了:", sorted(p.name for p in (CLONE_DIR / 'src').iterdir()))

## 2. 実行

In [ ]:
from src import config, data, features, model

print(config.describe())  # DATA_RAW が /kaggle/input/titanic になっていることを確認

In [ ]:
train = data.load_train()
test = data.load_test()
X, y, X_test = features.build(train, test)
oof, models = model.run_cv(X, y)

## 3. 提出ファイル

`/kaggle/working/submission.csv` に出力すると、右上の **Submit** から提出できる。

In [ ]:
import pandas as pd

proba = model.predict(models, X_test)
pd.DataFrame({
    config.ID_COL: test[config.ID_COL],
    config.TARGET: (proba > 0.5).astype(int),
}).to_csv("/kaggle/working/submission.csv", index=False)
print("submission.csv を書き出しました")